In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import numpy as np
import pandas as pd
import h5py

from hapROH.utils.miscellanious import print_memory_usage

In [3]:
def inspect_hdf5(item):
    """Recursively print all fields in the hdf5 with their shape and dtype"""
    if isinstance(item, h5py.Dataset):
        print(item.name, item.shape, item.dtype)
    else:
        for child in item.values():
            inspect_hdf5(child)

In [4]:
prefix_refHDF5 = "../Data/1000gMAF5.hdf5/maf5_chr"

sample_name = "SB606"
path_bam = f"../Data/ExampleWGS_bam/{sample_name}.merged.rg.markdup.indrealn_recalibrated.bam"
dir_sampleHDF5 = f"../Data/ExampleHDF5/"

## Bam to hdf5

In [5]:
from hapROH.utils.bam2hdf5 import bam2hdf5s

In [6]:
bam2hdf5s(path_bam, prefix_refHDF5, dir_sampleHDF5, sample_name, overwrite=False)

File ../Data/ExampleHDF5/SB606.chr1.hdf5 exists already and overwrite=False. Nothing happend.
File ../Data/ExampleHDF5/SB606.chr2.hdf5 exists already and overwrite=False. Nothing happend.
File ../Data/ExampleHDF5/SB606.chr3.hdf5 exists already and overwrite=False. Nothing happend.
File ../Data/ExampleHDF5/SB606.chr4.hdf5 exists already and overwrite=False. Nothing happend.
File ../Data/ExampleHDF5/SB606.chr5.hdf5 exists already and overwrite=False. Nothing happend.
File ../Data/ExampleHDF5/SB606.chr6.hdf5 exists already and overwrite=False. Nothing happend.
File ../Data/ExampleHDF5/SB606.chr7.hdf5 exists already and overwrite=False. Nothing happend.
File ../Data/ExampleHDF5/SB606.chr8.hdf5 exists already and overwrite=False. Nothing happend.
File ../Data/ExampleHDF5/SB606.chr9.hdf5 exists already and overwrite=False. Nothing happend.
File ../Data/ExampleHDF5/SB606.chr10.hdf5 exists already and overwrite=False. Nothing happend.
File ../Data/ExampleHDF5/SB606.chr11.hdf5 exists already an

In [7]:
chrom = 1
path_sample = os.path.join(dir_sampleHDF5, f"{sample_name}.chr{chrom}.hdf5")
path_ref = prefix_refHDF5 + f"{chrom}.hdf5"

with h5py.File(path_ref, "r") as h5_file:
    inspect_hdf5(h5_file)

print()

with h5py.File(path_sample, "r") as h5_file:
    inspect_hdf5(h5_file)

/calldata/GT (530434, 2504, 2) int8
/samples (2504,) object
/variants/ALT (530434,) object
/variants/MAP (530434,) float32
/variants/POS (530434,) int32
/variants/REF (530434,) object

/calldata/AD (63250, 1, 2) int64
/samples (1,) |S50
/variants/ALT (63250,) |S1
/variants/CHROM (63250,) int64
/variants/MAP (63250,) float64
/variants/POS (63250,) int64
/variants/REF (63250,) |S1


## GeneticDataFile

In [8]:
from hapROH.classes.genomicData import GenomicDataFile

In [9]:
file_sample = GenomicDataFile.load_genetic_file(path_sample)
file_ref = GenomicDataFile.load_genetic_file(path_ref)

In [10]:
print_memory_usage()
data_ref = file_ref.get_data()
print_memory_usage()

Memory usage: 137,469,952
Memory usage: 2,937,114,624


In [11]:
print(data_ref.data.shape, data_ref.data.nbytes)

(530434, 2504, 2) 2656413472


In [ ]:
idx_row = np.random.choice(range(530434), 63250, replace=False)
idx_row.sort()

idx_row_mask = np.zeros(530434, dtype=bool)
idx_row_mask[idx_row] = True

print_memory_usage()
my_new_data = file_ref.get_data(idx_row_mask)
print_memory_usage()    # why such an increase ? first call to get_data so some h5py gets loaded ?

Memory usage: 2,939,023,360
Memory usage: 8,325,513,216


## Call ROH

In [13]:
from hapROH.run_new import callROH_chr

In [14]:
post_pb = callROH_chr(path_sample, path_ref, chrom, sample_name, logfile=None, loglevel=2)

22:27:37.595 [INFO] hapROH.run_new: Starting callROH_chr on chromosome 1
22:27:37.597 [DEBUG] hapROH.run_new: Memory usage: 8,325,591,040


22:27:37.601 [INFO] hapROH.run_new: Loading SNP and computing intersection
22:27:37.629 [DEBUG] hapROH.classes.genomicData: Kept 63250/63250 biallelic SNP sites
22:27:38.110 [DEBUG] hapROH.classes.genomicData: Kept 530434/530434 biallelic SNP sites
22:27:38.117 [DEBUG] hapROH.classes.genomicData: Field `chrom` not found in `snp_ref`
22:27:38.158 [DEBUG] hapROH.classes.genomicData: 63250/530434 SNP found in intersection, of which 0 flipped REF/ALT and 0 mismatching REF/ALT
22:27:38.168 [INFO] hapROH.run_new: Found 63250 intersecting SNP, of which 0 flipped REF/ALT
22:27:38.170 [DEBUG] hapROH.run_new: Memory usage: 8,326,885,376
22:27:38.175 [INFO] hapROH.run_new: Minimum Genetic Map: 0.02012999914586544 Morgan
22:27:38.176 [INFO] hapROH.run_new: Maximum Genetic Map: 2.862730026245117 Morgan
22:27:38.177 [INFO] hapROH.run_new: Gaps bigger than 0.1 cM: 362
22:27:38.177 [INFO] hapROH.run_new: Maximum Gap: 2.4319 cM
22:27:38.178 [INFO] hapROH.run_new: Clipping gaps to range: 0.000 - inf cM


In [15]:
np.ones((3, 63250, 1), dtype=float).nbytes

1518000